In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────
SHAPE_LOG       = r"C:/ClaudeCode/research/01. regime_identification/input/shape_log.csv"
ENRICHED_LOG    = r"C:/ClaudeCode/research/03. validation_analysis/shape_log_enriched.csv"
STOCK           = r"C:/ClaudeCode/Raw Data/Stock and Production/FCPO Stock 3Y.xlsx"
PRODUCTION      = r"C:/ClaudeCode/Raw Data/Stock and Production/MPOB Production 3Y.xlsx"
EXPORT          = r"C:/ClaudeCode/Raw Data/Stock and Production/MPOB Export 3Y.xlsx"
USDMYR          = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/FX_IDC_USDMYR, 1D_1227e.csv"
PALMSOY         = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/MYX_DLY_FCPO1!_2_CBOT_DL_ZL1!, 1D_61912.csv"
CRUDE           = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/NYMEX_DL_CL1!, 1D_84001.csv"
ENSO_ASCII      = r"C:/ClaudeCode/Raw Data/ENSO/oni.ascii.txt"
LOG_FILE        = r"C:/ClaudeCode/research/03. validation_analysis/further_validation_log.txt"

# ── shape name mapping ─────────────────────────────────────────
SHAPE_NAMES = {
    '0.0': 'SB  (Steep Backwardation)',
    '0.1': 'MB  (Mild Backwardation)',
    '0.2': 'TB  (Transitional Backwardation)',
    '1':   'C   (Contango)',
    '2':   'F   (Flat/Back-Inversion)',
}
SHAPE_SHORT = {
    '0.0': 'SB', '0.1': 'MB',
    '0.2': 'TB', '1': 'C', '2': 'F',
}

def display_shape(code):
    return SHAPE_NAMES.get(str(code), str(code))

def display_short(code):
    return SHAPE_SHORT.get(str(code), str(code))

def write_log(study_id, content):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    header = f"""
{"="*70}
{study_id}
Run: {timestamp}
{"="*70}
"""
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(header + content + "\n")
    print(f"[LOG] Written -- {study_id}")

TRAIN_END  = '2024-12-31'
TEST_START = '2025-01-01'

print("Setup complete.")
print("Shape name mapping:")
for k, v in SHAPE_NAMES.items():
    print(f"  {k} -> {v}")

Setup complete.
Shape name mapping:
  0.0 -> SB  (Steep Backwardation)
  0.1 -> MB  (Mild Backwardation)
  0.2 -> TB  (Transitional Backwardation)
  1 -> C   (Contango)
  2 -> F   (Flat/Back-Inversion)


In [2]:
# ── load shape log (enriched version with duration + prior) ───
df_enriched = pd.read_csv(
    ENRICHED_LOG,
    dtype={'shape': str, 'prior_shape': str},
    parse_dates=['date'])
df_enriched = df_enriched[
    df_enriched['date'] >= '2017-01-01'
].sort_values('date').reset_index(drop=True)

# ── load MPOB ──────────────────────────────────────────────────
def load_mpob(path, col_name):
    df = pd.read_excel(path, parse_dates=True)
    date_col = df.columns[0]
    val_col  = df.columns[1]
    df = df.rename(columns={date_col: 'date', val_col: col_name})
    df['date'] = pd.to_datetime(df['date'])
    df = df.dropna(subset=['date'])
    return df[['date', col_name]].sort_values('date')

stock = load_mpob(STOCK,      'stock_raw')
prod  = load_mpob(PRODUCTION, 'production_raw')
exp   = load_mpob(EXPORT,     'export_raw')

# ── load macro (CSV with Unix timestamps) ─────────────────────
def load_macro(path, col_name):
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['time'], unit='s')
    df = df.rename(columns={'close': col_name})
    return df[['date', col_name]].sort_values('date')

usdmyr  = load_macro(USDMYR,   'usd_myr')
palmsoy = load_macro(PALMSOY,  'palm_soy')
crude   = load_macro(CRUDE,    'crude_oil')

# ── load ENSO (3-month season format) ─────────────────────────
enso = pd.read_csv(ENSO_ASCII, sep=r'\s+',
    names=['seas','year','total','oni'], skiprows=1)
seas_month = {
    'DJF':1, 'JFM':2, 'FMA':3, 'MAM':4,
    'AMJ':5, 'MJJ':6, 'JJA':7, 'JAS':8,
    'ASO':9, 'SON':10, 'OND':11, 'NDJ':12,
}
enso['month'] = enso['seas'].map(seas_month)
enso['date'] = pd.to_datetime(
    enso[['year','month']].assign(day=1))
enso_long = enso[['date','oni']].dropna().sort_values('date')

# ── merge everything onto enriched shape log ───────────────────
df = df_enriched.copy()

for src, col in [(stock,'stock_raw'),
                  (prod, 'production_raw'),
                  (exp,  'export_raw')]:
    df = pd.merge_asof(
        df.sort_values('date'),
        src.sort_values('date'),
        on='date', direction='backward')

df = pd.merge_asof(
    df.sort_values('date'),
    enso_long.sort_values('date'),
    on='date', direction='backward')

for src, col in [(usdmyr,'usd_myr'),
                  (palmsoy,'palm_soy'),
                  (crude,  'crude_oil')]:
    df = pd.merge_asof(
        df.sort_values('date'),
        src.sort_values('date'),
        on='date', direction='backward')

# ── derived variables ──────────────────────────────────────────
df['stock_pct']        = df['stock_raw'].rank(pct=True)
df['prod_mom_1m']      = df['production_raw'].pct_change(21)*100
df['prod_mom_3m']      = df['production_raw'].pct_change(63)*100
df['prod_yoy']         = df['production_raw'].pct_change(252)*100
df['export_yoy']       = df['export_raw'].pct_change(252)*100
df['usd_myr_chg_4w']   = df['usd_myr'].pct_change(20)*100
df['palm_soy_chg_4w']  = df['palm_soy'].pct_change(20)*100
df['crude_chg_4w']     = df['crude_oil'].pct_change(20)*100
df['month']            = df['date'].dt.month

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['prior_shape_enc'] = le.fit_transform(
    df['prior_shape'].fillna('unknown'))

df['stock_state'] = np.where(
    df['stock_pct'] >= df['stock_pct'].median(),
    'High', 'Low')
df['prod_state'] = np.where(
    df['prod_mom_3m'] >= 0, 'Rising', 'Falling')
df['stock_prod_quad'] = df['stock_state'] + '_' + df['prod_state']

quad_map = {
    'High_Falling': 3,
    'Low_Rising':   2,
    'Low_Falling':  1,
    'High_Rising':  0,
}
df['stock_prod_interaction'] = df['stock_prod_quad'].map(
    quad_map).fillna(0)

print(f"Panel built: {len(df)} rows")
print(f"Date range: {df['date'].min().date()} to "
      f"{df['date'].max().date()}")
print(f"\nMissing values in key columns:")
key_cols = ['stock_pct','prod_mom_3m','oni',
            'usd_myr_chg_4w','palm_soy_chg_4w',
            'prior_shape_enc','days_in_shape']
print(df[key_cols].isnull().sum())

Panel built: 2268 rows
Date range: 2017-01-02 to 2026-06-26

Missing values in key columns:
stock_pct           0
prod_mom_3m        63
oni                 0
usd_myr_chg_4w     20
palm_soy_chg_4w    20
prior_shape_enc     0
days_in_shape       0
dtype: int64


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

# ── feature set ───────────────────────────────────────────────
FEATURES = [
    'stock_pct', 'prod_mom_3m', 'prod_yoy',
    'export_yoy', 'oni', 'usd_myr_chg_4w',
    'palm_soy_chg_4w', 'days_in_shape',
    'prior_shape_enc', 'month',
    'stock_prod_interaction',
]

# ── build targets ──────────────────────────────────────────────
df_tm = df.sort_values('date').reset_index(drop=True)

# 1-week target (+5 trading days)
df_tm['target_1w'] = df_tm['shape'].shift(-5)

# 2-week target (+14 calendar days, ~10 trading days)
df_tm['target_2w'] = df_tm['shape'].shift(-10)

# Drop rows without both targets
df_tm = df_tm.dropna(subset=FEATURES + ['target_1w','target_2w'])

# Train/test split
df_train = df_tm[df_tm['date'] <= TRAIN_END].copy()
df_test  = df_tm[df_tm['date'] >= TEST_START].copy()

print(f"Train: {len(df_train)} rows")
print(f"Test:  {len(df_test)} rows")
print(f"\n1-week target distribution (train):")
for s, n in df_train['target_1w'].value_counts().items():
    print(f"  {display_short(s)}: {n}")
print(f"\n2-week target distribution (train):")
for s, n in df_train['target_2w'].value_counts().items():
    print(f"  {display_short(s)}: {n}")

Train: 1656 rows
Test:  350 rows

1-week target distribution (train):
  SB: 665
  C: 534
  MB: 218
  F: 148
  TB: 91

2-week target distribution (train):
  SB: 670
  C: 534
  MB: 218
  F: 143
  TB: 91


In [4]:
def fit_transition_model(current_shape, df_train,
                          df_test, features, target_col,
                          horizon_label):
    """
    Fits a per-shape transition model.
    Returns result dict with model, accuracy,
    baseline, and probability outputs.
    """
    tr = df_train[df_train['shape'] == current_shape
                  ].dropna(subset=features+[target_col])
    te = df_test[ df_test['shape']  == current_shape
                  ].dropna(subset=features+[target_col])

    result = {
        'shape': current_shape,
        'horizon': horizon_label,
        'train_n': len(tr),
        'test_n':  len(te),
    }

    if len(tr) < 30:
        result['status'] = 'INSUFFICIENT DATA'
        return result

    X_tr = tr[features].values
    y_tr = tr[target_col].values
    X_te = te[features].values if len(te) >= 5 else None
    y_te = te[target_col].values if len(te) >= 5 else None

    # Unconditional baseline — most common next shape
    from collections import Counter
    baseline_shape = Counter(y_tr).most_common(1)[0][0]
    baseline_acc   = (y_tr == baseline_shape).mean()*100

    # Unconditional distribution
    trans_dist = (pd.Series(y_tr)
                    .value_counts(normalize=True)*100
                    ).round(1).to_dict()

    model = RandomForestClassifier(
        n_estimators=300, max_depth=5,
        min_samples_leaf=8, random_state=42,
        class_weight='balanced')

    cv = cross_val_score(
        model, X_tr, y_tr,
        cv=StratifiedKFold(min(5, len(tr)//10)),
        scoring='accuracy').mean()*100

    model.fit(X_tr, y_tr)

    result['baseline_acc'] = round(baseline_acc, 1)
    result['trans_dist']   = trans_dist
    result['cv_acc']       = round(cv, 1)
    result['model']        = model
    result['classes']      = list(model.classes_)
    result['features']     = features

    # Feature importance
    result['feature_imp'] = dict(zip(
        features, model.feature_importances_.round(3)))

    if X_te is not None:
        y_pred   = model.predict(X_te)
        test_acc = accuracy_score(y_te, y_pred)*100
        result['test_acc'] = round(test_acc, 1)
        result['vs_base']  = round(test_acc - baseline_acc, 1)
        result['y_te']     = y_te
        result['X_te']     = X_te
        result['status']   = 'FITTED'
    else:
        result['test_acc'] = None
        result['status']   = 'FITTED (no test data)'

    return result

# ── fit all shapes at both horizons ───────────────────────────
shapes = sorted(df_tm['shape'].unique())
results_1w = {}
results_2w = {}

print("Fitting 1-week models...")
for s in shapes:
    results_1w[s] = fit_transition_model(
        s, df_train, df_test, FEATURES,
        'target_1w', '1-week')
    print(f"  {display_short(s)}: "
          f"{results_1w[s].get('status','?')}")

print("\nFitting 2-week models...")
for s in shapes:
    results_2w[s] = fit_transition_model(
        s, df_train, df_test, FEATURES,
        'target_2w', '2-week')
    print(f"  {display_short(s)}: "
          f"{results_2w[s].get('status','?')}")

Fitting 1-week models...


  SB: FITTED


  MB: FITTED


  TB: FITTED


  C: FITTED


...

  SB: FITTED


  MB: FITTED


  TB: FITTED


  C: FITTED


  F: FITTED


In [5]:
output_lines = [
    "TRANSITION MODEL — ACCURACY COMPARISON",
    "1-WEEK (+5 days) vs 2-WEEK (+14 days)",
    "="*70,
    f"\n{'Shape':<6} {'1w Base':>8} {'1w CV':>7} "
    f"{'1w Test':>8} {'1w vs':>7} "
    f"{'2w Base':>8} {'2w CV':>7} "
    f"{'2w Test':>8} {'2w vs':>7}"
]
print(output_lines[-1])
print("-"*80)

for s in shapes:
    r1 = results_1w[s]
    r2 = results_2w[s]

    def fmt(r, key, suffix='%'):
        v = r.get(key)
        return f"{v:.1f}{suffix}" if v is not None else "—"

    line = (f"  {display_short(s):<4} "
            f"{fmt(r1,'baseline_acc'):>8} "
            f"{fmt(r1,'cv_acc'):>7} "
            f"{fmt(r1,'test_acc'):>8} "
            f"{fmt(r1,'vs_base',suffix='pp'):>7} "
            f"{fmt(r2,'baseline_acc'):>8} "
            f"{fmt(r2,'cv_acc'):>7} "
            f"{fmt(r2,'test_acc'):>8} "
            f"{fmt(r2,'vs_base',suffix='pp'):>7}")
    print(line)
    output_lines.append(line)

write_log("FURTHER STUDY 2 — ACCURACY COMPARISON",
          "\n".join(output_lines))


Shape   1w Base   1w CV  1w Test   1w vs  2w Base   2w CV  2w Test   2w vs
--------------------------------------------------------------------------------
  SB      85.2%   62.7%    55.0% -30.2pp    80.2%   59.2%    56.2% -23.9pp
  MB      56.6%   44.2%    44.0% -12.6pp    47.5%   54.3%    24.0% -23.5pp
  TB      54.9%   49.4%    14.3% -40.7pp    34.1%   56.0%    14.3% -19.8pp
  C       89.7%   81.8%    57.7% -32.0pp    86.5%   79.4%    61.3% -25.3pp
  F       55.3%   37.5%    33.9% -21.4pp    45.4%   35.6%    29.9% -15.5pp
[LOG] Written -- FURTHER STUDY 2 — ACCURACY COMPARISON


In [6]:
def calibration_check(model, X_te, y_te, classes,
                       shape_label, horizon):
    """
    Tests whether high model probability = higher
    actual occurrence rate.
    Returns list of calibration results per
    next-shape transition.
    """
    probs = model.predict_proba(X_te)
    prob_df = pd.DataFrame(probs, columns=classes)
    prob_df['actual'] = list(y_te)
    results = []

    for next_s in classes:
        sub = prob_df[[next_s,'actual']].copy()
        sub['predicted'] = sub[next_s]
        sub['occurred']  = (sub['actual']==next_s).astype(int)

        median_prob = sub['predicted'].median()
        high_mask   = sub['predicted'] > median_prob
        low_mask    = ~high_mask

        if high_mask.sum() < 3 or low_mask.sum() < 3:
            continue

        base_rate  = sub['occurred'].mean()*100
        high_rate  = sub[high_mask]['occurred'].mean()*100
        low_rate   = sub[low_mask]['occurred'].mean()*100
        gap        = high_rate - low_rate

        results.append({
            'current':    shape_label,
            'next':       next_s,
            'horizon':    horizon,
            'base_rate':  round(base_rate, 1),
            'high_rate':  round(high_rate, 1),
            'low_rate':   round(low_rate, 1),
            'gap':        round(gap, 1),
            'n_high':     int(high_mask.sum()),
            'n_low':      int(low_mask.sum()),
            'calibrated': ('YES'     if gap > 5
                           else 'PARTIAL' if gap > 2
                           else 'NO'),
        })
    return results

# ── run calibration on both horizons ──────────────────────────
all_cal_1w = []
all_cal_2w = []

print("CALIBRATION CHECK — 1-WEEK HORIZON")
print("="*70)
output_1w = ["CALIBRATION CHECK — 1-WEEK HORIZON", "="*70]

for s in shapes:
    r = results_1w[s]
    if r.get('status') != 'FITTED' or 'model' not in r:
        continue
    if r.get('X_te') is None:
        continue

    cal = calibration_check(
        r['model'], r['X_te'], r['y_te'],
        r['classes'], s, '1w')
    all_cal_1w.extend(cal)

    line = f"\n  {display_shape(s)} (test n={r['test_n']}):"
    print(line); output_1w.append(line)
    line = (f"  {'Next Shape':<8} {'Base%':>7} "
            f"{'High%':>7} {'Low%':>7} "
            f"{'Gap':>6} {'Status':>10}")
    print(line); output_1w.append(line)

    for cr in sorted(cal, key=lambda x: -x['gap']):
        flag = " ***" if cr['calibrated']=='YES' else ""
        line = (f"    -> {display_short(cr['next']):<6} "
                f"{cr['base_rate']:>6.1f}%"
                f"{cr['high_rate']:>6.1f}% "
                f"{cr['low_rate']:>6.1f}% "
                f"{cr['gap']:>+5.1f}pp "
                f"{cr['calibrated']:>10}{flag}")
        print(line); output_1w.append(line)

write_log("FURTHER STUDY 2 — CALIBRATION 1-WEEK",
          "\n".join(output_1w))

print("\nCALIBRATION CHECK — 2-WEEK HORIZON")
print("="*70)
output_2w = ["CALIBRATION CHECK — 2-WEEK HORIZON", "="*70]

for s in shapes:
    r = results_2w[s]
    if r.get('status') != 'FITTED' or 'model' not in r:
        continue
    if r.get('X_te') is None:
        continue

    cal = calibration_check(
        r['model'], r['X_te'], r['y_te'],
        r['classes'], s, '2w')
    all_cal_2w.extend(cal)

    line = f"\n  {display_shape(s)} (test n={r['test_n']}):"
    print(line); output_2w.append(line)
    line = (f"  {'Next Shape':<8} {'Base%':>7} "
            f"{'High%':>7} {'Low%':>7} "
            f"{'Gap':>6} {'Status':>10}")
    print(line); output_2w.append(line)

    for cr in sorted(cal, key=lambda x: -x['gap']):
        flag = " ***" if cr['calibrated']=='YES' else ""
        line = (f"    -> {display_short(cr['next']):<6} "
                f"{cr['base_rate']:>6.1f}%"
                f"{cr['high_rate']:>6.1f}% "
                f"{cr['low_rate']:>6.1f}% "
                f"{cr['gap']:>+5.1f}pp "
                f"{cr['calibrated']:>10}{flag}")
        print(line); output_2w.append(line)

write_log("FURTHER STUDY 2 — CALIBRATION 2-WEEK",
          "\n".join(output_2w))

CALIBRATION CHECK — 1-WEEK HORIZON

  SB  (Steep Backwardation) (test n=80):
  Next Shape   Base%   High%    Low%    Gap     Status
    -> SB       82.5%  92.5%   72.5% +20.0pp        YES ***
    -> TB        3.8%   7.5%    0.0%  +7.5pp        YES ***
    -> C         3.8%   7.5%    0.0%  +7.5pp        YES ***
    -> MB        8.8%  10.0%    7.5%  +2.5pp    PARTIAL
    -> F         1.2%   0.0%    1.5%  -1.5pp         NO

  MB  (Mild Backwardation) (test n=25):
  Next Shape   Base%   High%    Low%    Gap     Status
    -> F        28.0%  58.3%    0.0% +58.3pp        YES ***
    -> SB       20.0%  41.7%    0.0% +41.7pp        YES ***
    -> C         4.0%   8.3%    0.0%  +8.3pp        YES ***
    -> TB        8.0%   0.0%   15.4% -15.4pp         NO
    -> MB       40.0%  16.7%   61.5% -44.9pp         NO

  TB  (Transitional Backwardation) (test n=7):
  Next Shape   Base%   High%    Low%    Gap     Status
    -> TB       14.3%  33.3%    0.0% +33.3pp        YES ***
    -> C        14.3%  33


  TB  (Transitional Backwardation) (test n=7):


  Next Shape   Base%   High%    Low%    Gap     Status
    -> C        28.6%  33.3%   25.0%  +8.3pp        YES ***
    -> TB        0.0%   0.0%    0.0%  +0.0pp         NO
    -> F         0.0%   0.0%    0.0%  +0.0pp         NO
    -> MB       14.3%   0.0%   25.0% -25.0pp         NO
    -> SB       57.1%  33.3%   75.0% -41.7pp         NO

  C   (Contango) (test n=111):
  Next Shape   Base%   High%    Low%    Gap     Status
    -> C        64.0%  70.9%   57.1% +13.8pp        YES ***
    -> TB        1.8%   3.6%    0.0%  +3.6pp    PARTIAL
    -> F        33.3%  34.5%   32.1%  +2.4pp    PARTIAL
    -> MB        0.9%   1.8%    0.0%  +1.8pp         NO
    -> SB        0.0%   0.0%    0.0%  +0.0pp         NO

  F   (Flat/Back-Inversion) (test n=127):
  Next Shape   Base%   High%    Low%    Gap     Status
    -> MB        7.1%  11.1%    3.1%  +8.0pp        YES ***
    -> F        60.6%  61.9%   59.4%  +2.5pp    PARTIAL
    -> SB        0.0%   0.0%    0.0%  +0.0pp         NO
    -> TB        0.

In [7]:
cal_1w_yes = sum(1 for c in all_cal_1w
                 if c['calibrated']=='YES')
cal_2w_yes = sum(1 for c in all_cal_2w
                 if c['calibrated']=='YES')

output_lines = [
    "TRANSITION MODEL — OVERALL VERDICT",
    "="*70,
    f"\n1-week calibrated cells: {cal_1w_yes} of "
    f"{len(all_cal_1w)}",
    f"2-week calibrated cells: {cal_2w_yes} of "
    f"{len(all_cal_2w)}",
    "\nBEST CALIBRATED EDGES (1-week):"
]
print("\n".join(output_lines))

best_1w = sorted(
    [c for c in all_cal_1w if c['calibrated']=='YES'],
    key=lambda x: -x['gap'])
for c in best_1w:
    line = (f"  {display_short(c['current'])} -> "
            f"{display_short(c['next'])}: "
            f"gap={c['gap']:+.1f}pp  "
            f"base={c['base_rate']:.1f}%  "
            f"high={c['high_rate']:.1f}%")
    print(line); output_lines.append(line)

output_lines.append("\nBEST CALIBRATED EDGES (2-week):")
best_2w = sorted(
    [c for c in all_cal_2w if c['calibrated']=='YES'],
    key=lambda x: -x['gap'])
for c in best_2w:
    line = (f"  {display_short(c['current'])} -> "
            f"{display_short(c['next'])}: "
            f"gap={c['gap']:+.1f}pp  "
            f"base={c['base_rate']:.1f}%  "
            f"high={c['high_rate']:.1f}%")
    print(line); output_lines.append(line)

# ── today's live read ──────────────────────────────────────────
latest = df_tm.sort_values('date').dropna(
    subset=FEATURES).iloc[-1]
current_s = latest['shape']

output_lines.append(
    f"\nTODAY'S LIVE READ — {latest['date'].date()}")
output_lines.append(
    f"Current shape: {display_shape(current_s)}")

for horizon, results in [('1-WEEK', results_1w),
                          ('2-WEEK', results_2w)]:
    r = results.get(current_s, {})
    if r.get('status') != 'FITTED' or 'model' not in r:
        continue

    model   = r['model']
    X_today = latest[FEATURES].values.reshape(1,-1)
    probs   = model.predict_proba(X_today)[0]
    classes = model.classes_

    line = f"\n  {horizon} OUTLOOK:"
    print(line); output_lines.append(line)

    # Check calibration for each transition
    cal_lookup = {
        (c['current'], c['next']): c
        for c in (all_cal_1w if horizon=='1-WEEK'
                  else all_cal_2w)
    }

    for shape_label, prob in sorted(
            zip(classes, probs), key=lambda x: -x[1]):
        unconditional = r['trans_dist'].get(
            shape_label, 0)
        edge = prob*100 - unconditional
        cal  = cal_lookup.get(
            (current_s, shape_label), {})
        cal_status = cal.get('calibrated', '?')
        use_flag = ("USE"    if cal_status=='YES'
                    else "SOFT"   if cal_status=='PARTIAL'
                    else "IGNORE" if cal_status=='NO'
                    else "?")
        bar = "\u2588" * int(prob*25)
        line = (f"    {display_short(shape_label):<4} "
                f"{prob*100:>5.1f}%  {bar:<25}  "
                f"unconditional={unconditional:.1f}%  "
                f"edge={edge:+.1f}pp  {use_flag}")
        print(line); output_lines.append(line)

write_log("FURTHER STUDY 2 — VERDICT AND LIVE READ",
          "\n".join(output_lines))
print("\n[NOTEBOOK 2 COMPLETE]")

TRANSITION MODEL — OVERALL VERDICT

1-week calibrated cells: 12 of 25
2-week calibrated cells: 10 of 25

BEST CALIBRATED EDGES (1-week):
  MB -> F: gap=+58.3pp  base=28.0%  high=58.3%
  MB -> SB: gap=+41.7pp  base=20.0%  high=41.7%
  TB -> TB: gap=+33.3pp  base=14.3%  high=33.3%
  TB -> C: gap=+33.3pp  base=14.3%  high=33.3%
  SB -> SB: gap=+20.0pp  base=82.5%  high=92.5%
  C -> F: gap=+16.8pp  base=33.3%  high=41.8%
  C -> C: gap=+8.4pp  base=64.9%  high=69.1%
  MB -> C: gap=+8.3pp  base=4.0%  high=8.3%
  F -> MB: gap=+8.0pp  base=5.5%  high=9.5%
  SB -> TB: gap=+7.5pp  base=3.8%  high=7.5%
  SB -> C: gap=+7.5pp  base=3.8%  high=7.5%
  F -> F: gap=+7.3pp  base=63.0%  high=66.7%
  MB -> F: gap=+59.0pp  base=40.0%  high=77.8%
  MB -> SB: gap=+41.7pp  base=20.0%  high=41.7%
  SB -> SB: gap=+32.5pp  base=76.2%  high=92.5%
  MB -> TB: gap=+16.7pp  base=8.0%  high=16.7%
  C -> C: gap=+13.8pp  base=64.0%  high=70.9%
  SB -> C: gap=+10.0pp  base=5.0%  high=10.0%
  MB -> C: gap=+9.0pp  base=12